In [1]:
from llmcompressor import oneshot
from llmcompressor.modifiers.pruning.sparsegpt import SparseGPTModifier

modifier_kwargs = {
    "targets": ["Linear"],
    "ignore": ["lm_head"],
    "sparsity": 0.2,
}
#modifier_kwargs["mask_structure"] = "2:4"

recipe = [SparseGPTModifier(**modifier_kwargs)]

c:\Users\lapud\anaconda3\envs\llm_compressor\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-28T17:04:23.6379 | get_main_device | WARNING - No accelerator available! Compressing model on CPU instead


In [3]:
from huggingface_hub import login

login()

In [4]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
model_path = "meta-llama/Llama-2-7b-chat-hf"
device = "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    device_map=device,
    )

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]c:\Users\lapud\anaconda3\envs\llm_compressor\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lapud\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\lapud\anaconda3\envs\llm_compressor\Lib\site-packages\huggingface_hub

In [5]:
from datasets import load_dataset, Dataset

traindata = load_dataset('allenai/c4', data_files={'train': 'en/c4-train.00000-of-01024.json.gz'}, split='train')

In [ ]:
oneshot(
    model=model,
    tokenizer=tokenizer,
    dataset=traindata, #calib_data,
    recipe=recipe,
    max_seq_length=2048,
    num_calibration_samples=1024,
)

2026-09-28T19:45:22.4483 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.


Tokenizing: 100%|██████████| 356317/356317 [15:27<00:00, 384.00 examples/s] 

2026-09-28T20:00:51.2718 | reset | INFO - Compression lifecycle reset
2026-09-28T20:00:51.3907 | from_modifiers | INFO - Creating recipe from modifiers
2026-09-28T20:00:51.4100 | get_main_device | WARNING - No accelerator available! Compressing model on CPU instead


2026-09-28T20:00:51.5134 | get_main_device | WARNING - No accelerator available! Compressing model on CPU instead
2026-09-28T20:00:51.5650 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-09-28T20:00:51.5672 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `SparseGPTModifier`
2026-09-28T20:00:51.5683 | get_main_device | WARNING - No accelerator available! Compressing model on CPU instead


W0928 20:00:51.861000 27884 site-packages\torch\fx\_symbolic_trace.py:57] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(2/33): Calibrating:   6%|▌         | 60/1024 [34:46<2:10:04,  8.10s/it]  